# Example: Eigendecomposition of Covariance and Feature Gram Matrices
In this example, we'll decompose both the empirical covariance matrix $\hat{\mathbf{\Sigma}}$ and a feature Gram matrix $\mathbf{C}$ into their eigenvalues and eigenvectors. We'll explore how modeling interactions between features (via a kernel function) changes the eigendecomposition compared to linear covariance, and how this gives us insight into how kernel methods capture nonlinear structure in the data.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Compute and decompose the empirical covariance matrix:__ Construct the centering matrix $\mathbf{H}$, compute the centered data matrix $\tilde{\mathbf{X}}$, and perform an eigendecomposition of the annualized empirical covariance matrix $\hat{\mathbf{\Sigma}}$ for S&P 500 return data.
> * __Build and decompose a feature Gram matrix:__ Evaluate a squared exponential kernel function on all pairs of features to construct the feature Gram matrix $\mathbf{C} \in \mathbb{R}^{m \times m}$, and compute its eigendecomposition.
> * __Compare linear and nonlinear similarity rankings:__ Interpret the eigenvectors of the covariance matrix and feature Gram matrix to rank firms by their influence, and compare how linear versus nonlinear methods group firms.


Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

### Data
We gathered a daily open-high-low-close dataset for each firm in the S&P 500 from `01-03-2014` until `12-31-2024`, along with data for a few exchange-traded funds and volatility products during that time. 

Let's load the `original_dataset::DataFrame` by calling [the `MyTrainingMarketDataSet()` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyTrainingMarketDataSet) and remove firms that do not have the maximum number of trading days. The cleaned dataset $\mathcal{D}$ will be stored in the `dataset` variable.

In [2]:
original_dataset = MyTrainingMarketDataSet() |> x-> x["dataset"];

Not all tickers in our dataset have the maximum number of trading days for various reasons, e.g., acquisition or de-listing events. Let's collect only those tickers with the maximum number of trading days.

First, let's compute the number of records for a firm that we know has a maximum value, e.g., `AAPL`, and save that value in the `maximum_number_trading_days::Int64` variable:

In [3]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow;

Now, let's iterate through our data and collect only tickers with `maximum_number_trading_days` records. Save that data in the `dataset::Dict{String,DataFrame}` variable:

In [4]:
dataset = let

    dataset = Dict{String,DataFrame}();
    for (ticker,data) ∈ original_dataset
        if (nrow(data) == maximum_number_trading_days)
            dataset[ticker] = data;
        end
    end
    dataset
end;

Finally, let's get a list of the firms in our cleaned dataset (and sort them alphabetically). We store the sorted firm ticker symbols in the `list_of_tickers::Array{String,1}` variable.

In [5]:
list_of_tickers = keys(dataset) |> collect |> sort # list of firm "ticker" symbols in alphabetical order

424-element Vector{String}:
 "A"
 "AAL"
 "AAP"
 "AAPL"
 "ABBV"
 "ABT"
 "ACN"
 "ADBE"
 "ADI"
 "ADM"
 ⋮
 "WYNN"
 "XEL"
 "XOM"
 "XRAY"
 "XYL"
 "YUM"
 "ZBRA"
 "ZION"
 "ZTS"

Finally, let's set up a ticker map that holds the index of each ticker value. We'll save this in the `tickerindexmap::Dict{String,Int}` dictionary:

In [6]:
tickerindexmap = let

    # initialize -
    tickerindexmap = Dict{String,Int}();
    for i ∈ eachindex(list_of_tickers)
        tickerindexmap[list_of_tickers[i]] = i;
    end

    tickerindexmap;
end;

__Load the tickers file:__ The tickers file (contained in the `data/` folder) contains a list of all S&P 500 firms as of `12-31-2024`, along with a few exchange-traded funds and volatility products, along with the full name of each firm and the business sector for each firm. We'll save data for each ticker in the `tickerinfo::Dict{String, NamedTuple}` dictionary (where the keys are the ticker symbols, and the values are named tuples containing the full name `name` and business sector `sector` of each firm):

In [7]:
tickerinfo = let

    # initialize -
    tickerinfo = Dict{String, NamedTuple}();
    
    # load the ticker data file -
    path_to_ticker_data_file = joinpath(_PATH_TO_DATA, "Ticker-data.csv");
    df = CSV.read(path_to_ticker_data_file, DataFrame; stringtype=String);
    number_of_firms = nrow(df);

    for i ∈ 1:number_of_firms
        ticker = df[i, :Symbol];
        name = df[i, :Name] |> String;
        sector = df[i, :Sector] |> String;
        tickerinfo[ticker] = (name=name, sector=sector);
    end

    tickerinfo;
end

Dict{String, NamedTuple} with 515 entries:
  "NI"   => (name = "NiSource", sector = "Utilities")
  "EMR"  => (name = "Emerson Electric Company", sector = "Industrials")
  "CTAS" => (name = "Cintas Corporation", sector = "Industrials")
  "HSIC" => (name = "Henry Schein", sector = "Health Care")
  "KIM"  => (name = "Kimco Realty", sector = "Real Estate")
  "PLD"  => (name = "Prologis", sector = "Real Estate")
  "IEX"  => (name = "IDEX Corporation", sector = "Industrials")
  "TPR"  => (name = "Tapestry", sector = "Consumer Discretionary")
  "BAC"  => (name = "Bank of America", sector = "Financials")
  "CBOE" => (name = "Cboe Global Markets", sector = "Financials")
  "EXR"  => (name = "Extra Space Storage", sector = "Real Estate")
  "NCLH" => (name = "Norwegian Cruise Line Holdings", sector = "Consumer Discre…
  "CVS"  => (name = "CVS Health", sector = "Health Care")
  "DRI"  => (name = "Darden Restaurants", sector = "Consumer Discretionary")
  "DTE"  => (name = "DTE Energy", sector = "Uti

### Compute the return matrix
Next, let's compute the return array which contains, for each day and each firm in our dataset, the value of the growth rate between time $j$ and $j-1$. 

>  __Continuously Compounded Growth Rate (CCGR)__
>
> Let's assume a model of the share price of firm $i$ is governed by an expression of the form:
>$$
\begin{align*}
S^{(i)}_{j} &= S^{(i)}_{j-1}\;\exp\left(\underbrace{g^{(i)}_{j,j-1}\Delta{t}_{j}}_{\text{return}}\right)
\end{align*}
$$
> where $S^{(i)}_{j-1}$ denotes the share price of firm $i$ at time index $j-1$, $S^{(i)}_{j}$ denotes the share price of firm $i$ at time index $j$, and $\Delta{t}_{j} = t_{j} - t_{j-1}$ denotes the length of a time step (units: years) between time index $j-1$ and $j$. The value we are going to estimate is the return, which is the growth rate $g^{(i)}_{j,j-1}$ (units: inverse years) for each firm $i$ multiplied by the time step in the dataset. 

We've implemented [the `log_growth_matrix(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.log_growth_matrix) which takes the cleaned dataset and a list of ticker symbols, and returns the growth rate array. Each row of the growth rate array is a time step, while each column corresponds to a firm from the `list_of_tickers::Array{String,1}` array. We can get the return by passing in a time-step value of `1`.

In [8]:
X = let

    # initialize -
    r̄ = 0.0; # assume the risk-free rate is 0

    # compute the growth matrix -
    growth_rate_array = log_growth_matrix(dataset, list_of_tickers, Δt = 1.0, 
        risk_free_rate = r̄); # other optional parameters are at their defaults

    growth_rate_array; # return
end;

___

## Task 1: Compute and Decompose Empirical Covariance Matrix
In this task, let's compute the (annualized) empirical covariance matrix $\hat{\mathbf{\Sigma}}$ for our dataset $\mathcal{D}$ using code that we write ourselves (we'll never do this in practice, but it's a good exercise). The annualized empirical covariance matrix is given by:
$$
\hat{\mathbf{\Sigma}} = \frac{T}{n-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}
$$
where $T = 252$ denotes the number of trading days per year (the annualization factor), and $\tilde{\mathbf{X}}$ is the centered data matrix:
$$
\tilde{\mathbf{X}} = \mathbf{X} - \mathbf{1}\mathbf{m}^{\top}
$$
where $\mathbf{1} \in \mathbb{R}^{n}$ is a vector of ones, and $\mathbf{1}\mathbf{m}^{\top}$ creates an $n \times m$ matrix where each row is identical and contains the mean returns for each firm. 

> __Outer product:__ The $\mathbf{1}\mathbf{m}^{\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a} \in \mathbb{R}^{n}$ and $\mathbf{b} \in \mathbb{R}^{m}$ is the $n \times m$ matrix $\mathbf{a}\mathbf{b}^{\top}$. Each element of the outer product is computed as $(\mathbf{a}\mathbf{b}^{\top})_{ij} = a_i b_j$. 

Alternatively, if we define the centering matrix $\mathbf{H} = \mathbf{I}_n - \frac{1}{n}\mathbf{1}\mathbf{1}^\top$, we can express this more compactly as $\tilde{\mathbf{X}} = \mathbf{H}\mathbf{X}$. We'll use this matrix form in the code below. First, let's start by computing the centering matrix $\mathbf{H}$ and then use it to center the data matrix $\mathbf{X}$. 

We'll store the centering matrix in the `H::Array{Float64,2}` variable:

In [9]:
H = let
    
    # initialize -
    n, m = size(X); # n = number of time periods, m = number of firms
    I_n = Matrix{Float64}(I, n, n); # identity matrix of size n x n
    one_vector = ones(n); # vector of ones of size n
    H = I_n .- (1/n) .* (one_vector * transpose(one_vector)); # centering matrix

    H; # return
end;

Next, let's compute the centered data matrix $\tilde{\mathbf{X}}$ by multiplying the centering matrix $\mathbf{H}$ with the data matrix $\mathbf{X}$. We'll store the centered data matrix in the `X̃::Array{Float64,2}` variable:

In [10]:
X̃ = H * X; # centered data matrix

__Wow!__ That seems a little magical, let's check that the mean of each column of `X̃` is (approximately) zero. We can do this by computing the mean of each column and using an `@assert` statement to check that the absolute value of each mean is less than a small tolerance value (e.g., `1e-10`). 

> __What is the `all(...)` function?__ The [`all(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.all-Tuple{AbstractArray,%20Any}) in Julia checks if all elements of a collection satisfy a given condition. It returns `true` if every element meets the condition and `false` otherwise. In our case, we use it to verify that all column means are approximately zero.

So what do we see?

In [11]:
let

    # initialize -
    number_of_columns = size(X̃, 2); # number of firms (columns)
    m = mean(X̃, dims=1); # mean of each column
    ϵ = 1e-10; # tolerance for checking if mean is approximately zero

    # let's check that the mean of each column is approximately zero using @assert, and the all(...) function -
    @assert all( abs.(m) .< ϵ ) "Error: The mean of one or more columns is not approximately zero!"
end

Now that we have the centered data matrix, let's compute the empirical covariance matrix $\hat{\mathbf{\Sigma}}$ and store it in the `Σ̂::Array{Float64,2}` variable:

In [12]:
Σ̂ = let

    # initialize -
    T = 252; # total number of trading days
    n = size(X̃, 1); # number of time periods (rows)

    # compute the empirical covariance matrix -
    Σ̂ = (T/(n-1)) .* (transpose(X̃) * X̃); # empirical covariance matrix (annualized)

    Σ̂; # return
end;

In [13]:
Σ̂

424×424 Matrix{Float64}:
 0.0535771   0.032397    0.0212982  …  0.0355662   0.0280938   0.0266666
 0.032397    0.207093    0.0431437     0.051429    0.0689041   0.0280481
 0.0212982   0.0431437   0.117391      0.0308539   0.037867    0.019717
 0.0229702   0.0312627   0.0163174     0.032633    0.0201827   0.022609
 0.0179206   0.0176175   0.0158407     0.0157326   0.0168026   0.0185472
 0.0244078   0.0195811   0.0145372  …  0.023159    0.0167416   0.0221966
 0.025643    0.0334402   0.0218931     0.0313218   0.0280694   0.023824
 0.0294242   0.02952     0.0185233     0.0369478   0.0198333   0.0262802
 0.0297618   0.0458278   0.0231343     0.0433011   0.0335821   0.0233085
 0.0169025   0.0325409   0.0206063     0.0219846   0.0327002   0.0131991
 ⋮                                  ⋱                          
 0.0339199   0.0921199   0.035255   …  0.0503196   0.0581754   0.0308858
 0.00888225  0.00814595  0.0118987     0.00616705  0.00592554  0.0112662
 0.0161836   0.0376862   0.0190661    

Finally, let's compute the eigendecomposition of the empirical covariance matrix $\hat{\mathbf{\Sigma}}$ and store the eigenvalues in the `λ::Array{Float64,1}` variable and the eigenvectors in the `V::Array{Float64,2}` variable using [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen). 

[The `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) returns a structure containing the eigenvalues and eigenvectors, which we can access using the `.values` and `.vectors` fields, respectively.

In [14]:
(λ, V) = let

    # compute the eigendecomposition using the built-in function -
    F = eigen(Σ̂);
    λ = F.values; # grab the eigenvalues
    V = F.vectors; # grab the eigenvectors

    # sort the eigenpairs by eigenvalue magnitude -
    p = sortperm(λ, rev=true); # indices that would sort λ in descending order (largest to smallest)
    λ = λ[p] # sort the eigenvalues in descending order
    V = V[:,p] # sort the eigenvectors according to the sorted order of the eigenvalues
    
    (λ, V); # return the sorted eigenvalues and eigenvectors
end

([11.438058870655455, 1.595819459020512, 1.1983172323103417, 0.9913463704158774, 0.8821003299986337, 0.7218934238657474, 0.5110538829534114, 0.4433158835074157, 0.424100945759947, 0.40109541051294584  …  0.003141200093345497, 0.0030032234323236085, 0.0029450466978899175, 0.002904341968560427, 0.002437486055628757, 0.0023239405869121313, 0.0011683475444739587, 0.0008197993962320261, 0.0008004082795630409, 0.00020091649390590935], [-0.04241385029420439 -0.057850524592016436 … 0.0037493839419421833 -0.002294131234357775; -0.08444908198854559 0.05786956337900914 … -0.0006976040595419945 0.0009776709554534398; … ; -0.06871769022824514 0.07600622038329019 … -0.00197865349177016 -0.008092901031749863; -0.037098632606339624 -0.06134369996581138 … 0.006274549177537995 -0.0019203225901286416])

Let's explore the eigenvectors of $\hat{\mathbf{\Sigma}}$ to identify which firms have the largest influence. We normalize the absolute values of each eigenvector's components so they sum to one, then rank firms by their normalized weight. This reveals which firms dominate each principal direction of variance in the return data.

In [15]:
let
    
    # initialize -
    number_of_firms = length(list_of_tickers);
    number_of_firms_to_display = 10; # number of firms to display
    df = DataFrame(); # initialize the DataFrame to hold the results
    index_to_explore = 3; # index of the eigenvector to explore (1 for the largest eigenvector, 2 for the second largest, etc.)

    # get the largest eigenvector, and scale it appropriately
    v₁ = V[:,index_to_explore]; # get the largest eigenvector (associated with the largest eigenvalue)
    σ = sum(abs.(v₁)) |> T -> (1/T)*abs.(v₁)
    sortperm_indices = sortperm(σ, rev=true); # indices that would sort σ in descending order
    
    
    for i ∈ 1:number_of_firms_to_display
        firm_index = sortperm_indices[i];
        ticker = list_of_tickers[firm_index];
        influence = σ[firm_index];
        
        # get data from the tickerinfo dictionary
        name = tickerinfo[ticker].name
        sector = tickerinfo[ticker].sector
        
        push!(df, (
            Ticker = ticker, 
            rank = i, 
            Name = name, 
            Sector = sector
        ));
    end

    # make a table display of the results -
    pretty_table(
        df;
        backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );

end

 -------- ------- -------------------------- ------------------------
  Ticker    rank                       Name                   Sector 
  String   Int64                     String                   String 
 -------- ------- -------------------------- ------------------------
    ENPH       1             Enphase Energy   Information Technology
     AMD       2     Advanced Micro Devices   Information Technology
    NVDA       3                     Nvidia   Information Technology
      MU       4          Micron Technology   Information Technology
    MPWR       5   Monolithic Power Systems   Information Technology
     CZR       6      Caesars Entertainment   Consumer Discretionary
     WEC       7           WEC Energy Group                Utilities
      ED       8        Consolidated Edison                Utilities
    AMAT       9          Applied Materials   Information Technology
     LNT      10             Alliant Energy                Utilities
 -------- ------- ------------

___

## Task 2: Eigendecomposition of the feature matrix $\mathbf{C}$
In this task, we'll specify a kernel function, compute the feature Gram matrix $\mathbf{C}$ for our dataset $\mathcal{D}$ by applying the kernel to all pairs of features, and then compute the eigendecomposition of $\mathbf{C}$. We'll compare the eigenvalues and eigenvectors of $\mathbf{C}$ to those of the empirical covariance matrix $\hat{\mathbf{\Sigma}}$ that we computed in task 1, and discuss how the kernel captures nonlinear structure in the data.

> **Note:** We're computing the feature Gram matrix $\mathbf{C} \in \mathbb{R}^{m \times m}$ (kernel applied to pairs of features), not the sample kernel matrix $\mathbf{K} \in \mathbb{R}^{n \times n}$ used in standard kernel methods. This measures nonlinear similarity between features rather than between samples.

Let's start by defining a kernel function. We'll use the functions exported by [the `KernelFunctions.jl` package](https://juliagaussianprocesses.github.io/KernelFunctions.jl/stable/#KernelFunctions.jl) where we our choice of kernel function as `k`:

In [16]:
k = SqExponentialKernel(); # squared exponential kernel function

Now that we have a kernel function, let's compute the feature Gram matrix $\mathbf{C}$ for our dataset $\mathcal{D}$. The feature Gram matrix is computed by evaluating the kernel function on all pairs of features (columns) in our dataset. We'll store the feature Gram matrix in the `C::Array{Float64,2}` variable:

In [17]:
C = let

    # initialize -
    (number_of_rows, number_of_columns) = size(X̃); # number of time periods (rows) and number of firms (columns)
    C = zeros(Float64, number_of_columns, number_of_columns); # initialize the feature Gram matrix
    D = X̃; # the data matrix (not centered)

    # compute the feature Gram matrix -
    for i ∈ 1:number_of_columns
        for j ∈ 1:number_of_columns
            C[i,j] = k(D[:,i], D[:,j]); # evaluate the kernel function on the i-th and j-th columns of X
        end
    end
    
    C; # return
end

424×424 Matrix{Float64}:
 1.0       0.341435  0.494473  0.709198  …  0.585699  0.524822  0.759336
 0.341435  1.0       0.270692  0.334594     0.30026   0.353765  0.332088
 0.494473  0.270692  1.0       0.46454      0.391901  0.411661  0.495765
 0.709198  0.334594  0.46454   1.0          0.562745  0.477452  0.720633
 0.656256  0.28175   0.45198   0.615458     0.457243  0.449977  0.674104
 0.782675  0.319754  0.494884  0.722603  …  0.550972  0.499454  0.779313
 0.767386  0.360084  0.518922  0.749984     0.582872  0.547042  0.767385
 0.677077  0.291964  0.423302  0.689894     0.524793  0.423038  0.667304
 0.714294  0.367003  0.468007  0.718073     0.591417  0.517042  0.678881
 0.666118  0.340648  0.48883   0.633714     0.502652  0.549889  0.652486
 ⋮                                       ⋱                      
 0.375985  0.306718  0.268837  0.371362  …  0.321231  0.340562  0.371005
 0.689527  0.294631  0.502206  0.683021     0.477653  0.463347  0.722069
 0.648722  0.3538    0.471793  0.6

Now, let's decompose the feature Gram matrix $\mathbf{C}$ using [the `eigen(...)` function](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/#LinearAlgebra.eigen) and store the eigenvalues in the `λ_k::Array{Float64,1}` variable and the eigenvectors in the `V_k::Array{Float64,2}` variable:

In [18]:
(λ_k, V_k) = let

    # compute the eigendecomposition using the built-in function -
    F = eigen(C); # eigendecomposition of the feature Gram matrix
    λ = F.values; # grab the eigenvalues
    V = F.vectors; # grab the eigenvectors

    # sort the eigenpairs by eigenvalue magnitude -
    p = sortperm(λ, rev=true); # indices that would sort λ in descending order (largest to smallest)
    λ = λ[p] # sort the eigenvalues in descending order
    V = V[:,p] # sort the eigenvectors according to the sorted order of the eigenvalues
    
    (λ, V); # return the sorted eigenvalues and eigenvectors
end;

Now let's repeat the same eigenvector analysis for the feature Gram matrix $\mathbf{C}$. We normalize the absolute values of each eigenvector's components and rank firms by their weight. Comparing these rankings to those from the covariance matrix reveals how the nonlinear kernel captures different structure in the data.

In [19]:
let
    
    # initialize -
    number_of_firms = length(list_of_tickers);
    number_of_firms_to_display = 10; # number of firms to display
    df = DataFrame(); # initialize the DataFrame to hold the results
    index_to_explore = 3; # index of the eigenvector to explore (1 for the largest eigenvector, 2 for the second largest, etc.)

    # get the largest eigenvector, and scale it appropriately
    v₁ = V_k[:,index_to_explore]; # get the largest eigenvector (associated with the largest eigenvalue)
    σ = sum(abs.(v₁)) |> T -> (1/T)*abs.(v₁)
    sortperm_indices = sortperm(σ, rev=true); # indices that would sort σ in descending order
    
    
    for i ∈ 1:number_of_firms_to_display
        firm_index = sortperm_indices[i];
        ticker = list_of_tickers[firm_index];
        influence = σ[firm_index];
        
        # get data from the tickerinfo dictionary
        name = tickerinfo[ticker].name
        sector = tickerinfo[ticker].sector
        
        push!(df, (
            Ticker = ticker, 
            rank = i, 
            Name = name, 
            Sector = sector
        ));
    end

    # make a table display of the results -
    pretty_table(
        df;
        backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact)
    );
end

 -------- ------- -------------------------- ------------------------
  Ticker    rank                       Name                   Sector 
  String   Int64                     String                   String 
 -------- ------- -------------------------- ------------------------
    MPWR       1   Monolithic Power Systems   Information Technology
    NVDA       2                     Nvidia   Information Technology
    AMAT       3          Applied Materials   Information Technology
    LRCX       4               Lam Research   Information Technology
    SNPS       5                   Synopsys   Information Technology
    CDNS       6     Cadence Design Systems   Information Technology
     NOW       7                 ServiceNow   Information Technology
     TER       8                   Teradyne   Information Technology
    KLAC       9            KLA Corporation   Information Technology
    ADBE      10                      Adobe   Information Technology
 -------- ------- ------------

___

## Summary
In this example, we computed the eigendecomposition of both the annualized empirical covariance matrix $\hat{\mathbf{\Sigma}}$ and a feature Gram matrix $\mathbf{C}$ constructed using a squared exponential kernel to rank S&P 500 firms by their influence along principal directions of variance.

> __Key Takeaways:__
> 
> * **Covariance eigenvectors capture linear co-movement:** The eigenvectors of $\hat{\mathbf{\Sigma}}$ identify firms whose returns move together linearly, revealing sector-level groupings driven by correlated price dynamics.
> * **Feature Gram matrix eigenvectors capture nonlinear similarity:** The feature Gram matrix $\mathbf{C}$ (constructed by applying the squared exponential kernel to all pairs of features) measures pairwise nonlinear similarity between features, and its eigenvectors can surface firm groupings that linear covariance analysis misses.
> * **Comparing rankings reveals structural differences:** Firms that rank highly under the covariance decomposition may not rank the same under the feature Gram matrix decomposition, highlighting how nonlinear interactions reshape the similarity structure.


These techniques provide a foundation for kernel-based dimensionality reduction and clustering methods that extend beyond linear assumptions.
___